# Hazard of invalidation — age vs distance to pivot

Supports FINDINGS.md section 2a. Run against committed data in this repo.

**Question.** The marginal hazard of invalidation falls with elapsed bars.
Is that a property of age, or a composition effect from old pivots sitting
further from the barrier?

**Pre-declared prediction.** The age gap collapses once distance is controlled.
Confirming band: under 30% remaining gap between the youngest and oldest age bin
within a distance band.

**Result.** Prediction failed. In the <1% band the hazard nearly doubles with age.


In [1]:
import pandas as pd, numpy as np, sys
sys.path.insert(0, "src")
from pivot_auto import find_pivots

df = pd.read_csv("data/btc_6h.csv", parse_dates=["date"])
p = find_pivots(df, atr_mult=3.0)
print(sorted(p.columns))

['atr_mult', 'bar', 'bars', 'bars_held', 'bars_to_35', 'date', 'gain', 'max_drop', 'max_up']


## 1. Survival, unconditional

`held = bars_held.isna()`, and `find_pivots` caps the forward scan at
`span_bars=32`. So the ~30% hold rate is a **32-bar survival rate**, not an
unconditional one.

In [2]:
b, n = p["bars_held"], len(p)
print(f"n={n}  held = {b.isna().sum()} ({b.isna().mean()*100:.0f}%)")
print(f"median bars_held (invalidators only) = {b.median()}")
print("k   cum_fail  at_risk  hazard")
for k in range(1, 33):
    at_risk = ((b >= k) | b.isna()).sum()
    haz = (b == k).sum() / at_risk if at_risk else np.nan
    print(f"{k:<4}{(b <= k).sum()/n:>9.3f}{at_risk:>9}{haz:>9.3f}")

n=231  held = 69 (30%)
median bars_held (invalidators only) = 7.0
k   cum_fail  at_risk  hazard
1       0.000      231    0.000
2       0.087      231    0.087
3       0.177      211    0.100
4       0.238      190    0.074
5       0.307      176    0.091
6       0.346      160    0.056
7       0.368      151    0.033
8       0.424      146    0.089
9       0.442      133    0.030
10      0.468      129    0.047
11      0.498      123    0.057
12      0.506      116    0.017
13      0.519      114    0.026
14      0.524      111    0.009
15      0.545      110    0.045
16      0.554      105    0.019
17      0.563      103    0.019
18      0.567      101    0.010
19      0.576      100    0.020
20      0.593       98    0.041
21      0.606       94    0.032
22      0.606       91    0.000
23      0.623       91    0.044
24      0.628       87    0.011
25      0.628       86    0.000
26      0.649       86    0.058
27      0.662       81    0.037
28      0.671       78    0.026
29      

Hazard at k=1 is exactly 0.000. The confirming bar is lower than the pivot by
definition and cannot close above it. **So `bars_held` counts from the pivot bar,
not the confirming close** — a one-bar offset against any figure quoted from the
confirming close.

## 2. Bar-level panel, with index check

One row per pivot-bar at risk. `dist` is the gap from the prior close up to the
pivot price. The index check reconstructs invalidation independently and compares
it to the module's own `bars_held`.

In [3]:
close, SPAN, N = df["close"].values, 32, len(df)
rows, chk = [], []
for _, r in p.iterrows():
    b0, pp = int(r["bar"]), float(df["high"].iat[int(r["bar"])])
    for j in range(1, SPAN + 1):
        if b0 + j >= N: break
        prior = close[b0 + j - 1]
        ev = close[b0 + j] > pp
        rows.append((j, (pp - prior) / pp * 100.0, bool(ev)))
        if ev:
            chk.append((j, r["bars_held"])); break

P = pd.DataFrame(rows, columns=["age","dist","event"])
d = pd.DataFrame(chk, columns=["j","bars_held"]).dropna()
off = (d.j - d.bars_held)
print(f"index check: offset mode={int(off.mode()[0])}  agree={off.eq(off.mode()[0]).mean():.2f}  n={len(d)}")

index check: offset mode=0  agree=1.00  n=162


`agree = 1.00`, offset 0 — invalidation is **close above the pivot high**, and
the reconstruction matches `find_pivots` exactly.

## 3. Hazard by distance x age

In [4]:
P["dbin"] = pd.cut(P.dist, [-99,1,2,4,99], labels=["<1%","1-2%","2-4%",">4%"])
P["abin"] = pd.cut(P.age, [0,6,12,33], labels=["2-6","7-12","13-32"])

print("marginal hazard by age")
for a, g in P.groupby("abin", observed=True):
    print(f"  {a:<7}{g.event.sum():>5}/{len(g):<6}{g.event.mean():>8.3f}")

print("\nhazard by distance x age")
print(f"{'dist':<8}" + "".join(f"{a:>16}" for a in ["2-6","7-12","13-32"]))
for dlab, g in P.groupby("dbin", observed=True):
    line = f"{dlab:<8}"
    for a in ["2-6","7-12","13-32"]:
        s = g[g.abin == a]
        line += f"{(str(int(s.event.sum()))+'/'+str(len(s))):>9}" + (f"{s.event.mean():>7.3f}" if len(s) >= 20 else f"{'-':>7}")
    print(line)

marginal hazard by age
  2-6       80/1199     0.067
  7-12      37/798      0.046
  13-32     45/1830     0.025

hazard by distance x age
dist                 2-6            7-12           13-32
<1%        52/361  0.144    23/86  0.267   32/121  0.264
1-2%       21/410  0.051    9/163  0.055    8/220  0.036
2-4%        7/316  0.022    4/348  0.011    4/531  0.008
>4%         0/112  0.000    1/201  0.005    1/958  0.001


## 4. Reading, and what is not established

The marginal decay is composition: 958 of 1,830 bar-exposures at ages 13-32 sit
beyond 4%, against 112 of 1,199 at ages 2-6.

Within <1%, hazard runs 0.144 / 0.267 / 0.264. A pivot still loitering near its
price at bar 20 has spent twenty bars failing to drop.


---

## 5. Validation — pre-declared

Section 3 found hazard rising with age inside the <1% band (0.144 / 0.267 / 0.264)
on 86 and 121 observations, one ATR threshold, highs only, full sample.

**Pre-declared before running.** In the <1% band, hazard(13-32) / hazard(2-6)
stays at or above **1.50** across all four checks, with no sign reversal.
Below 1.50 on any check, or a reversal in any, and section 3 is treated as noise.

Four checks: cluster by pivot rather than by bar; a second ATR threshold (2.0);
the lows mirror; the driftless window (2025+).

In [5]:
import pandas as pd, numpy as np, sys
sys.path.insert(0, "src")
from pivot_auto import find_pivots
from pivot_lows import find_lows
rng = np.random.default_rng(0)

BASE = pd.read_csv("data/btc_6h.csv", parse_dates=["date"])
SPAN = 32

def panel(p, side, d):
    """One row per pivot-bar at risk. side=+1 highs, -1 lows."""
    c = d["close"].values; n = len(d)
    px = (d["high"] if side == 1 else d["low"]).values
    rows = []
    for k, r in p.iterrows():
        b0 = int(r["bar"]); pp = float(px[b0])
        for j in range(1, SPAN + 1):
            if b0 + j >= n: break
            dist = (pp - c[b0 + j - 1]) / pp * 100.0 * side
            ev = (c[b0 + j] > pp) if side == 1 else (c[b0 + j] < pp)
            rows.append((k, j, dist, bool(ev)))
            if ev: break
    P = pd.DataFrame(rows, columns=["pid", "age", "dist", "event"])
    P["dbin"] = pd.cut(P.dist, [-99, 1, 2, 4, 99], labels=["<1%", "1-2%", "2-4%", ">4%"])
    P["abin"] = pd.cut(P.age, [0, 6, 12, 33], labels=["2-6", "7-12", "13-32"])
    return P

near = lambda P: P[P.dbin == "<1%"]

def line(lab, P):
    n = near(P)
    y = n[n.abin == "2-6"].event.mean(); o = n[n.abin == "13-32"].event.mean()
    r = o / y if y else np.nan
    print(f"{lab:<30}{y:>8.3f}{o:>8.3f}{r:>8.2f}   n={(n.abin=='2-6').sum()}/{(n.abin=='13-32').sum()}")
    return r

print(f"{'check':<30}{'young':>8}{'old':>8}{'ratio':>8}")
H30 = panel(find_pivots(BASE, atr_mult=3.0), +1, BASE); line("highs atr3.0 (baseline)", H30)
H20 = panel(find_pivots(BASE, atr_mult=2.0), +1, BASE); line("highs atr2.0", H20)
L30 = panel(find_lows(BASE, atr_mult=3.0), -1, BASE);   line("lows atr3.0", L30)
D = BASE[BASE.date >= "2025-01-01"].reset_index(drop=True)
D30 = panel(find_pivots(D, atr_mult=3.0), +1, D);       line("highs atr3.0 driftless 2025+", D30)

check                            young     old   ratio
highs atr3.0 (baseline)          0.144   0.264    1.84   n=361/121
highs atr2.0                     0.131   0.257    1.96   n=738/179
lows atr3.0                      0.141   0.222    1.58   n=241/81
highs atr3.0 driftless 2025+     0.140   0.214    1.53   n=229/70


1.5334821428571428

### Clustered bootstrap

Rows are not independent: one pivot contributes a streak of them. Resample the
**pivots**, not the rows. If the 5th percentile of the ratio sits below 1.0, the
effect is not distinguishable from nothing once clustering is accounted for.

In [6]:
def boot(P, B=2000):
    n = near(P); pids = n.pid.unique()
    idx = {i: g for i, g in n.groupby("pid")}
    out = []
    for _ in range(B):
        g = pd.concat([idx[i] for i in rng.choice(pids, len(pids), replace=True)])
        y = g[g.abin == "2-6"].event.mean(); o = g[g.abin == "13-32"].event.mean()
        if y and y > 0: out.append(o / y)
    return np.percentile(out, [5, 50, 95])

lo, mid, hi = boot(H30)
print(f"baseline ratio   p5={lo:.2f}   p50={mid:.2f}   p95={hi:.2f}   (B=2000, clustered by pivot)")

baseline ratio   p5=1.36   p50=1.84   p95=2.55   (B=2000, clustered by pivot)


### 5a. Bin sensitivity — pre-declared

The bins in section 3 were chosen on the first pass, and the whole effect lives
in the first one. If it depends on those exact cuts it is an artifact.

**Pre-declared.** The ratio stays at or above **1.50** across near-band widths
0.5 / 0.75 / 1.0 / 1.5 / 2.0 percent, and across age cut points 8 / 10 / 12 / 16.
Cells under 40 observations are printed but not counted toward the verdict.

In [7]:
def grid(P, widths=(0.5, 0.75, 1.0, 1.5, 2.0), cuts=(8, 10, 12, 16)):
    print(f"{'width':>7}" + "".join(f"{'cut '+str(c):>14}" for c in cuts))
    for w in widths:
        n = P[P.dist < w]
        row = f"{w:>7.2f}"
        for c in cuts:
            y = n[n.age <= 6].event.mean()
            o = n[n.age > c].event.mean()
            no = (n.age > c).sum()
            r = o / y if y else float("nan")
            row += f"{r:>9.2f}" + (f"[{no}]" if no >= 40 else f"[{no}]*")
        print(row)
    print("  * cell under 40 obs, not counted")

print("HIGHS atr3.0"); grid(H30)
print("\nLOWS atr3.0");  grid(L30)

HIGHS atr3.0
  width         cut 8        cut 10        cut 12        cut 16
   0.50     1.96[48]     2.11[40]     1.98[35]*     1.91[26]*
   0.75     1.92[101]     2.05[83]     1.93[73]     1.72[58]
   1.00     1.92[166]     1.98[137]     1.84[121]     1.72[93]
   1.50     1.85[285]     1.93[233]     1.77[211]     1.71[163]
   2.00     1.28[471]     1.29[392]     1.24[341]     1.22[259]
  * cell under 40 obs, not counted

LOWS atr3.0
  width         cut 8        cut 10        cut 12        cut 16
   0.50     1.74[43]     2.08[36]*     1.96[30]*     2.08[18]*
   0.75     1.19[76]     1.45[62]     1.49[49]     1.31[30]*
   1.00     1.20[142]     1.47[106]     1.58[81]     1.55[55]
   1.50     1.18[272]     1.45[203]     1.47[155]     1.55[98]
   2.00     0.99[428]     1.16[338]     1.19[262]     1.29[164]
  * cell under 40 obs, not counted


In [8]:
D = BASE[BASE.date >= "2025-01-01"].reset_index(drop=True)
L30d = panel(find_lows(D, atr_mult=3.0), -1, D)
H30d = panel(find_pivots(D, atr_mult=3.0), +1, D)

print("LOWS atr3.0, 2025+"); grid(L30d)
print("\nHIGHS atr3.0, 2025+ (reference)"); grid(H30d)

lo, mid, hi = boot(L30d)
print(f"\nlows 2025+ bootstrap  p5={lo:.2f}  p50={mid:.2f}  p95={hi:.2f}")

LOWS atr3.0, 2025+
  width         cut 8        cut 10        cut 12        cut 16
   0.50     2.27[22]*     2.78[18]*     2.67[15]*     2.50[12]*
   0.75     1.44[37]*     1.77[30]*     1.85[23]*     1.68[19]*
   1.00     1.25[71]     1.68[53]     1.71[41]     1.64[35]*
   1.50     1.10[153]     1.47[114]     1.39[89]     1.49[65]
   2.00     0.95[247]     1.20[195]     1.18[151]     1.36[103]
  * cell under 40 obs, not counted

HIGHS atr3.0, 2025+ (reference)
  width         cut 8        cut 10        cut 12        cut 16
   0.50     2.08[24]*     2.00[20]*     1.67[18]*     1.88[16]*
   0.75     1.87[55]     1.90[45]     1.71[40]     1.75[36]*
   1.00     1.73[91]     1.77[77]     1.53[70]     1.63[57]
   1.50     1.65[155]     1.74[131]     1.50[120]     1.66[97]
   2.00     1.20[263]     1.24[218]     1.13[192]     1.31[149]
  * cell under 40 obs, not counted

lows 2025+ bootstrap  p5=0.92  p50=1.75  p95=2.96


In [9]:
for m in (1.5, 2.0, 2.5, 3.0):
    L = panel(find_lows(BASE, atr_mult=m), -1, BASE)
    n = L[(L.dist < 1.0)]
    y = n[n.age <= 6].event.mean(); o = n[n.age > 12].event.mean()
    lo, mid, hi = boot(L)
    print(f"lows atr{m}  ratio={o/y:.2f}  n={(n.age<=6).sum()}/{(n.age>12).sum()}  "
          f"boot p5={lo:.2f} p50={mid:.2f} p95={hi:.2f}")

lows atr1.5  ratio=1.37  n=891/316  boot p5=1.07 p50=1.36 p95=1.75
lows atr2.0  ratio=1.75  n=546/172  boot p5=1.29 p50=1.76 p95=2.38
lows atr2.5  ratio=1.72  n=360/117  boot p5=1.18 p50=1.74 p95=2.52
lows atr3.0  ratio=1.58  n=241/81  boot p5=0.98 p50=1.60 p95=2.49


In [ ]:
for m in (2.0, 2.5, 3.0):
    L = panel(find_lows(BASE, atr_mult=m), -1, BASE)
    n = L[L.dist < 1.0]
    y = n[n.age <= 6].event.mean(); o = n[n.age > 12].event.mean()
    lo, mid, hi = boot(L)
    print(f"lows atr{m}  ratio={o/y:.2f}  n={(n.age<=6).sum()}/{(n.age>12).sum()}  "
          f"boot p5={lo:.2f} p50={mid:.2f} p95={hi:.2f}")

print()
for m in (1.5, 2.0, 2.5, 3.0):
    H = panel(find_pivots(BASE, atr_mult=m), +1, BASE)
    n = H[H.dist < 1.0]
    y = n[n.age <= 6].event.mean(); o = n[n.age > 12].event.mean()
    lo, mid, hi = boot(H)
    print(f"highs atr{m}  ratio={o/y:.2f}  n={(n.age<=6).sum()}/{(n.age>12).sum()}  "
          f"boot p5={lo:.2f} p50={mid:.2f} p95={hi:.2f}")

lows atr2.0  ratio=1.75  n=546/172  boot p5=1.27 p50=1.76 p95=2.39
lows atr2.5  ratio=1.72  n=360/117  boot p5=1.16 p50=1.74 p95=2.51
lows atr3.0  ratio=1.58  n=241/81  boot p5=0.98 p50=1.59 p95=2.51



### 5b. Lows in the driftless window

Full-sample lows carry BTC's 165% rise, the confound documented in section 3.
This checks whether the lows result survives the 2025+ window on its own.

Expect thin cells: 2025+ is roughly a third of the sample.

In [ ]:
D = BASE[BASE.date >= "2025-01-01"].reset_index(drop=True)
L30d = panel(find_lows(D, atr_mult=3.0), -1, D)
H30d = panel(find_pivots(D, atr_mult=3.0), +1, D)

print("LOWS atr3.0, 2025+"); grid(L30d)
print("\nHIGHS atr3.0, 2025+ (reference)"); grid(H30d)

lo, mid, hi = boot(L30d)
print(f"\nlows 2025+ bootstrap  p5={lo:.2f}  p50={mid:.2f}  p95={hi:.2f}")

LOWS atr3.0, 2025+
  width         cut 8        cut 10        cut 12        cut 16
   0.50     2.27[22]*     2.78[18]*     2.67[15]*     2.50[12]*
   0.75     1.44[37]*     1.77[30]*     1.85[23]*     1.68[19]*
   1.00     1.25[71]     1.68[53]     1.71[41]     1.64[35]*
   1.50     1.10[153]     1.47[114]     1.39[89]     1.49[65]
   2.00     0.95[247]     1.20[195]     1.18[151]     1.36[103]
  * cell under 40 obs, not counted

HIGHS atr3.0, 2025+ (reference)
  width         cut 8        cut 10        cut 12        cut 16
   0.50     2.08[24]*     2.00[20]*     1.67[18]*     1.88[16]*
   0.75     1.87[55]     1.90[45]     1.71[40]     1.75[36]*
   1.00     1.73[91]     1.77[77]     1.53[70]     1.63[57]
   1.50     1.65[155]     1.74[131]     1.50[120]     1.66[97]
   2.00     1.20[263]     1.24[218]     1.13[192]     1.31[149]
  * cell under 40 obs, not counted


**Result.** Lows rise in the driftless window at width 1.0 (1.25 / 1.68 / 1.71 /
1.64 against 1.20 / 1.47 / 1.58 / 1.55 full sample), so drift was suppressing them.
But the clustered interval is [0.90, 2.87]: it includes 1.0. The lows result is
**not established on the driftless window alone**, and is carried by the full-sample
threshold sweep above.